**Data Science Capstone** <br>
Final GO/NOGO Model

SVM Model - 2/3 Datasources

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import wandb
import joblib
from sklearn.svm import SVC, LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_predict, GridSearchCV, train_test_split
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay, 
                             roc_curve, auc, RocCurveDisplay)
from sklearn.decomposition import PCA

In [13]:
# Initialize Weights & Biases
wandb.init(
    entity="mmcgee18-georgia-state-university",
    project="Final GO-NOGO Classifier", 
    name="Final-SVM-Run-v1",
    config={
        "model_type": "LinearSVC",
        "gridsearch_samples": 100000,
        "val_samples": 500000,
        "cv_folds": 5
    }
)
config = wandb.config

baseline/accuracy,▁
baseline/weighted_f1,▁
optimized/accuracy,▁
optimized/weighted_f1,▁
baseline/accuracy,0.96254
baseline/weighted_f1,0.96256
optimized/accuracy,0.98358
optimized/weighted_f1,0.98358


Data Collection/Preview

In [14]:
# Load and Merge Data
hpsa_df = pd.read_csv('GitHub Capstone/DS_Capstone_Group_1/data/v2_cleaned/hpsa_need_category_gb_v2.csv')
monte_df = pd.read_csv('GitHub Capstone/DS_Capstone_Group_1/data/v2_cleaned/monte_carlo_simulation_results.csv')

In [15]:
monte_df.insert(0, 'geoid', hpsa_df['geoid'])
df = hpsa_df.merge(monte_df, on="geoid")
df.head()

,geoid_tract,geoid,hpsa_designated,hpsa_need_category,hpsa_need_category_label,diabetes_pct_reduction_mean,diabetes_pct_reduction_p05,diabetes_pct_reduction_p95,high_bp_pct_reduction_mean,high_bp_pct_reduction_p05,high_bp_pct_reduction_p95,high_cholesterol_pct_reduction_mean,high_cholesterol_pct_reduction_p05,high_cholesterol_pct_reduction_p95,asthma_pct_reduction_mean,asthma_pct_reduction_p05,asthma_pct_reduction_p95,no_checkup_pct_reduction_mean,no_checkup_pct_reduction_p05,no_checkup_pct_reduction_p95
0,1001020100,1001,1.0,1,hpsa_designated_high_need,0.032444,0.015547,0.052138,0.094713,0.046653,0.151013,0.122421,0.062349,0.202158,0.024049,0.012419,0.037703,0.377668,0.208467,0.571981
1,1001020100,1001,1.0,1,hpsa_designated_high_need,0.040363,0.020514,0.062889,0.105712,0.051960,0.165784,0.112325,0.058663,0.171148,0.026366,0.012959,0.041388,0.383723,0.215596,0.577626
2,1001020100,1001,1.0,1,hpsa_designated_high_need,0.032769,0.016827,0.053078,0.092074,0.047792,0.146349,0.115946,0.063437,0.183649,0.023594,0.012725,0.036823,0.361315,0.197578,0.551564
3,1001020100,1001,1.0,1,hpsa_designated_high_need,0.030406,0.015351,0.048521,0.094191,0.048619,0.150773,0.127970,0.065797,0.198910,0.021047,0.011126,0.033213,0.380811,0.216822,0.572805
4,1001020100,1001,1.0,1,hpsa_designated_high_need,0.026067,0.012786,0.041901,0.083019,0.043511,0.132988,0.110945,0.058505,0.173601,0.022165,0.011927,0.034781,0.366114,0.210646,0.560115


Pipeline

In [16]:
# 2. Define Target Label
high_need_cats = [1, 3]
reduction_cols = [col for col in df.columns if "_mean" in col]
df["avg_reduction"] = df[reduction_cols].mean(axis=1)
median_reduction = df["avg_reduction"].median()

df["target"] = 0
df.loc[(df["hpsa_need_category"].isin(high_need_cats)) & 
       (df["avg_reduction"] > median_reduction), "target"] = 1

# 3. Setup Pipeline
leakage_cols = ["target", "avg_reduction", "hpsa_need_category"] + reduction_cols
X = df.drop(columns=leakage_cols)
y = df["target"]

remaining_numeric = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
remaining_categorical = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), remaining_numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), remaining_categorical)
])

svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LinearSVC(C=1.0, dual=False, max_iter=10000))
])

In [17]:
# 4. W&B Logging Functions

def log_all_metrics(model, X_data, y_true, y_pred, wandb_tag="baseline"):
    """Logs Accuracy, F1, Confusion Matrix, and ROC Curve."""
    # 1. Classification Report
    report = classification_report(y_true, y_pred, output_dict=True)
    wandb.log({
        f"{wandb_tag}/accuracy": report['accuracy'],
        f"{wandb_tag}/weighted_f1": report['weighted avg']['f1-score'],
        f"{wandb_tag}/report_table": wandb.Table(dataframe=pd.DataFrame(report).transpose().reset_index())
    })

    # 2. Confusion Matrix (Interactive & Static)
    wandb.log({f"Confusion_Matrix/Interactive_{wandb_tag}": wandb.plot.confusion_matrix(
        y_true=y_true, preds=y_pred, class_names=['Bad', 'Good'])})
    
    fig, ax = plt.subplots()
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=['Bad', 'Good'], ax=ax, cmap='Blues')
    wandb.log({f"Confusion_Matrix/Static_{wandb_tag}": wandb.Image(plt)})
    plt.close()

    # 3. ROC Curve
    y_scores = model.decision_function(X_data)
    fig, ax = plt.subplots()
    RocCurveDisplay.from_predictions(y_true, y_scores, name=f"ROC {wandb_tag}", ax=ax)
    wandb.log({f"ROC_Curve/{wandb_tag}": wandb.Image(plt)})
    plt.close()

def log_feature_importance(model, wandb_tag="baseline"):
    """Extracts and logs coefficients for LinearSVC."""
    preprocessor = model.named_steps['preprocessor']
    feature_names = preprocessor.get_feature_names_out()
    coeffs = model.named_steps['classifier'].coef_.flatten()
    
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': np.abs(coeffs),
        'Coefficient': coeffs
    }).sort_values(by='Importance', ascending=False)
    
    wandb.log({f"Feature_Importance/{wandb_tag}_Table": wandb.Table(dataframe=importance_df)})
    
    plt.figure(figsize=(10, 8))
    plt.barh(importance_df['Feature'][:20], importance_df['Importance'][:20], color='skyblue')
    plt.gca().invert_yaxis()
    plt.title(f"Top 20 Features - {wandb_tag}")
    wandb.log({f"Feature_Importance/{wandb_tag}_Plot": wandb.Image(plt)})
    plt.close()

def plot_and_log_pca_boundary(model, X, y, title, subset_size=5000, wandb_tag="baseline"):
    """Saves PCA Decision Boundary plot to W&B."""
    indices = np.random.choice(len(X), subset_size, replace=False)
    X_sample, y_sample = X.iloc[indices], y.iloc[indices]
    X_transformed = model.named_steps['preprocessor'].transform(X_sample)
    
    X_pca = PCA(n_components=2).fit_transform(X_transformed)
    vis_svc = SVC(kernel='rbf', C=1.0).fit(X_pca, y_sample)
    
    x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
    y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))
    Z = vis_svc.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    plt.figure(figsize=(10, 6))
    plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.3)
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_sample, cmap=plt.cm.coolwarm, edgecolors='k', s=20)
    plt.title(title)
    wandb.log({f"Decision_Boundary/{wandb_tag}": wandb.Image(plt)})
    plt.close()

In [18]:
# 5. Execute Baseline
print("Running Baseline...")
y_pred_base = cross_val_predict(svm_pipeline, X, y, cv=StratifiedKFold(config.cv_folds))
svm_pipeline.fit(X, y)

log_all_metrics(svm_pipeline, X, y, y_pred_base, wandb_tag="baseline")
log_feature_importance(svm_pipeline, wandb_tag="baseline")
plot_and_log_pca_boundary(svm_pipeline, X, y, "Baseline Boundary", wandb_tag="baseline")

Running Baseline...


Grid Search Optimization

In [19]:
# 1. Grid Search
X_train_sub, _, y_train_sub, _ = train_test_split(X, y, train_size=config.gridsearch_samples, stratify=y, random_state=42)

param_grid = {"classifier__C": [0.1, 1, 10], "classifier__max_iter": [5000]}
grid_search = GridSearchCV(svm_pipeline, param_grid, cv=3, n_jobs=-1) 
grid_search.fit(X_train_sub, y_train_sub)
best_model = grid_search.best_estimator_

wandb.config.update({"best_params": grid_search.best_params_})

In [21]:
# 2. Final Evaluation with Index Fix
print("Evaluating Final Model...")
X_val, _, y_val, _ = train_test_split(X, y, train_size=config.val_samples, stratify=y, random_state=42)

# Reset index to avoid KeyError: 0
X_val = X_val.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)

y_pred_opt = cross_val_predict(best_model, X_val, y_val, cv=3)

# Fit on full data for the final artifact
best_model.fit(X, y)

# Log Everything
log_all_metrics(best_model, X_val, y_val, y_pred_opt, wandb_tag="optimized")
log_feature_importance(best_model, wandb_tag="optimized")
plot_and_log_pca_boundary(best_model, X_val, y_val, title="Optimized Boundary", wandb_tag="optimized")

Evaluating Final Model...


In [22]:
# Save Model File to W&B
model_filename = "best_svm_pipeline_v1.joblib"
joblib.dump(best_model, model_filename)
wandb.save(model_filename)

artifact = wandb.Artifact('go-nogo-svm', type='model')
artifact.add_file(model_filename)
wandb.log_artifact(artifact)

wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


<Artifact go-nogo-svm>

In [23]:
# 3. Cleanup
print("Run Complete. Results synced to W&B.")
wandb.finish()

Run Complete. Results synced to W&B.


baseline/accuracy,▁
baseline/weighted_f1,▁
optimized/accuracy,▁▁
optimized/weighted_f1,▁▁
baseline/accuracy,0.96254
baseline/weighted_f1,0.96256
optimized/accuracy,0.98358
optimized/weighted_f1,0.98358
